# Lab 2: From Idea to First Result in 30 Minutes

**Before you start: go to Runtime → Change runtime type and select GPU.**

Last week you saw what deep learning can do. This week you are going to 
build something yourself.

The goal of this notebook is simple: take a small real-world dataset, 
train a classifier on it using transfer learning, and get a first result — 
all in under 30 minutes, most of which is waiting for training to finish.

This is the same approach you will use in your project. The dataset will 
be different, the classes will be different, and you may need a more 
sophisticated model — but the pipeline is identical to what you see here.

**What you will do:**
1. Load a small dataset of real images (cats, dogs, and horses)
2. Train a classifier using a pretrained ResNet-18 backbone
3. Evaluate it and look at where it goes wrong
4. See how quickly you can get a result with minimal code

After the notebook, you will fill in the project scoping worksheet 
with your group.

## Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import urllib.request
import zipfile

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. The dataset

We will use a small dataset of cats, dogs, and horses — 300 images per class, 
split into training and validation sets. This is a deliberately small dataset 
to make training fast, and to show that transfer learning works well even 
with limited data.

In your project, this is where you would load your own data. The rest of 
the pipeline stays exactly the same.

In [ ]:
# Download the dataset
import gdown
file_id = '1KDMC39ba1MAL83FLLoSVSJY2KOmFR1aj'
gdown.download(f'https://drive.google.com/uc?id={file_id}', 'data.zip', quiet=False)

with zipfile.ZipFile('data.zip', 'r') as z:
    z.extractall('.')

data_dir = Path('data')
print('Dataset contents:')
for split in ['train', 'val']:
    for cls in sorted((data_dir / split).iterdir()):
        n = len(list(cls.glob('*')))
        print(f'  {split}/{cls.name}: {n} images')

In [ ]:
# Transforms — same pattern as Lab 5
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Load datasets using ImageFolder (one folder per class)
train_dataset = datasets.ImageFolder(data_dir / 'train', transform=train_transform)
val_dataset   = datasets.ImageFolder(data_dir / 'val',   transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=2)

class_names = train_dataset.classes
num_classes = len(class_names)
print(f'Classes: {class_names}')
print(f'Training: {len(train_dataset)} images')
print(f'Validation: {len(val_dataset)} images')

In [ ]:
# Always visualise your data before training
def denormalise(tensor):
    mean = torch.tensor(IMAGENET_MEAN).view(3,1,1)
    std  = torch.tensor(IMAGENET_STD).view(3,1,1)
    return torch.clamp(tensor * std + mean, 0, 1)

images, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 6, figsize=(16, 6))
for i, ax in enumerate(axes.flat):
    img = denormalise(images[i]).permute(1, 2, 0).numpy()
    ax.imshow(img)
    ax.set_title(class_names[labels[i]], fontsize=9)
    ax.axis('off')
plt.suptitle('Sample training images', fontsize=12)
plt.tight_layout()
plt.show()

## 2. The model — transfer learning

We use a ResNet-18 pretrained on ImageNet. The model already knows how to 
detect edges, textures, shapes, and object parts. We just need to teach it 
to distinguish our three classes.

We replace the final layer (which originally predicted 1,000 ImageNet classes) 
with a new layer for our 3 classes, and train the whole network.

In [ ]:
# Load pretrained ResNet-18 and adapt for our task
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model    = model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {trainable:,}')
print(f'Final layer: {model.fc}')

## 3. Training

We train for 10 epochs. On this small dataset with a pretrained backbone, 
10 epochs is enough to get a strong result.

Watch the loss and accuracy numbers as training progresses. The validation 
accuracy tells you how well the model generalises to images it has not seen.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

NUM_EPOCHS = 10

for epoch in range(1, NUM_EPOCHS + 1):

    # --- Training ---
    model.train()
    train_loss = train_correct = train_total = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        train_loss    += loss.item() * images.size(0)
        train_correct += (logits.argmax(1) == labels).sum().item()
        train_total   += images.size(0)

    # --- Validation ---
    model.eval()
    val_loss = val_correct = val_total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            logits  = model(images)
            loss    = criterion(logits, labels)
            val_loss    += loss.item() * images.size(0)
            val_correct += (logits.argmax(1) == labels).sum().item()
            val_total   += images.size(0)

    scheduler.step()

    # Record
    tl = train_loss / train_total
    ta = train_correct / train_total
    vl = val_loss / val_total
    va = val_correct / val_total
    history['train_loss'].append(tl)
    history['train_acc'].append(ta)
    history['val_loss'].append(vl)
    history['val_acc'].append(va)

    print(f'Epoch {epoch:2d}/{NUM_EPOCHS} | '
          f'Train loss: {tl:.3f}, acc: {ta:.3f} | '
          f'Val loss: {vl:.3f}, acc: {va:.3f}')

print(f'\nBest validation accuracy: {max(history["val_acc"]):.3f}')

In [ ]:
# Plot loss curves
epochs = range(1, NUM_EPOCHS + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(epochs, history['train_loss'], label='Train')
ax1.plot(epochs, history['val_loss'],   label='Validation')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Loss curves'); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(epochs, history['train_acc'], label='Train')
ax2.plot(epochs, history['val_acc'],   label='Validation')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.set_title('Accuracy curves'); ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Where does it go wrong?

A single accuracy number does not tell you much. Let's look at the 
**confusion matrix** — which classes get confused with which — and at 
the images the model gets most wrong.

Understanding *where* a model fails is just as important as knowing 
how accurate it is. This is something you will need to do in your project.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Collect all predictions on the validation set
model.eval()
all_preds  = []
all_labels = []
all_probs  = []
all_images = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        logits = model(images)
        probs  = torch.softmax(logits, dim=1)
        preds  = logits.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())
        all_images.extend(images.cpu())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs  = np.array(all_probs)

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion matrix (validation set)')
plt.tight_layout()
plt.show()

print('Rows = true class, Columns = predicted class.')
print('Diagonal = correct predictions. Off-diagonal = mistakes.')

In [ ]:
# Show the images the model was most wrong about
# These are validation images where the model was very confident but incorrect
wrong_mask       = all_preds != all_labels
wrong_indices    = np.where(wrong_mask)[0]
wrong_confidence = np.array([
    all_probs[i][all_preds[i]] for i in wrong_indices
])
# Sort by confidence (most confidently wrong first)
sorted_wrong = wrong_indices[np.argsort(wrong_confidence)[::-1]]

n_show = min(8, len(sorted_wrong))
if n_show == 0:
    print('No mistakes on the validation set — perfect accuracy!')
else:
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    for i, ax in enumerate(axes.flat):
        if i >= n_show:
            ax.axis('off')
            continue
        idx  = sorted_wrong[i]
        img  = denormalise(all_images[idx]).permute(1, 2, 0).numpy()
        true = class_names[all_labels[idx]]
        pred = class_names[all_preds[idx]]
        conf = all_probs[idx][all_preds[idx]] * 100
        ax.imshow(img)
        ax.set_title(
            f'True: {true}\nPred: {pred} ({conf:.0f}% confident)',
            fontsize=9, color='red'
        )
        ax.axis('off')
    plt.suptitle('Most confidently wrong predictions', fontsize=12)
    plt.tight_layout()
    plt.show()
    print(f'Total mistakes: {wrong_mask.sum()} / {len(all_labels)} '
          f'({wrong_mask.mean()*100:.1f}% error rate)')

## 5. What did we just do?

Let's take stock. In about 30 minutes you:

- Loaded a real dataset of images
- Adapted a model pretrained on 1.2 million images to a new 3-class problem
- Trained it for 10 epochs
- Got a validation accuracy likely above 95%
- Visualised where it goes wrong

All with about 50 lines of actual code.

**The key insight:** you did not need to understand convolutional neural 
networks, backpropagation, or any deep learning theory to get this result. 
Transfer learning lets you start from a strong foundation and adapt it to 
your problem.

The lectures over the coming weeks will explain *why* this works and *how* 
to make it work better. But the core pipeline — data, model, train, evaluate — 
is already in your hands.

---

## 6. Your project dataset

The second half of this lab session is for your group to work on the 
**project scoping worksheet**. Before you do that, spend 10 minutes 
answering these questions together:

1. **Could you frame your project idea as a classification problem?** 
   What would the classes be?

2. **Where would you get the data?** Do you know of an existing dataset, 
   or would you need to collect your own?

3. **How much data do you think you need?** As a rough guide, transfer 
   learning can work with as few as 50–100 images per class for simple 
   problems. Harder problems need more.

4. **What would 95% accuracy actually mean for your problem?** 
   Is that good enough, or do you need to do much better?

Take notes — these answers feed directly into the project scoping worksheet.